# SemanticSCVI hyperparameter sweep — Monocytes (GenePT prior)

Parallel to `four_way_benchmark_genept.ipynb`, but compares **multiple SemanticSCVI variants** side-by-side in a single report (no LDVAE+/scHPF/cNMF baselines).

**Config (cell 1):** edit `BASELINE` for the fixed defaults, and `SWEEP` for per-axis alternatives. Each value in `SWEEP[axis]` creates one variant that differs from baseline **only** in that axis. The baseline itself is always included. Duplicates are deduped.

Allowed sweep axes: `SEMANTIC_WARMUP_EPOCHS`, `SEMANTIC_N_EPOCHS_KL_WARMUP`, `coherence_weight`. `HVG_TOP_N` and `PER_PROJECTION_N_TOP` live in `BASELINE` only (they're shared across the single report).

In [ ]:
from pathlib import Path

def _find_nb_dir() -> Path:
    for p in [Path.cwd(), *Path.cwd().parents]:
        for cand in (
            p / "notebooks" / "benchmark_helpers.py",
            p / "benchmark_helpers.py",
            p / "scvi-tools-neural-nmf" / "notebooks" / "benchmark_helpers.py",
        ):
            if cand.exists():
                return cand.parent.resolve()
    raise FileNotFoundError(f"Could not locate benchmark_helpers.py from cwd={Path.cwd()}.")

NB_DIR = _find_nb_dir()

ADATA_PATH         = NB_DIR / "monocytes_clean.h5ad"
SEMANTIC_CACHE     = NB_DIR / "monocytes_clean_genept_3large.pt"
GENEPT_PICKLE_PATH = NB_DIR / "GenePT_gene_protein_embedding_model_3_text.pickle"
PATHWAY_INDEX      = NB_DIR / "pathway_index.pkl"
LIB1_GMT           = NB_DIR / "lib1_immune.gmt"
LIB2_GMT           = NB_DIR / "lib2_monocyte.gmt"
OUT_DIR            = NB_DIR / "benchmark_results" / "four_way_genept_sweep"
OUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_CACHE_DIR    = NB_DIR / ".model_cache"
SEM_CACHE_PARENT   = "semantic_scvi_genept_sweep"

# ---- Baseline: fixed defaults for every variant ----
BASELINE = {
    "SEMANTIC_WARMUP_EPOCHS":      40,
    "SEMANTIC_N_EPOCHS_KL_WARMUP": 80,
    "coherence_weight":            2000.0,
    "HVG_TOP_N":                   50,   # shared, not sweepable
    "PER_PROJECTION_N_TOP":        30,     # shared, not sweepable
}

# ---- Sweep: per-axis alternatives. Each value -> one variant differing only in this axis. ----
# Empty dict = baseline only. Only training axes allowed here.
SWEEP = {
    "SEMANTIC_WARMUP_EPOCHS":      [40, 80],
    "SEMANTIC_N_EPOCHS_KL_WARMUP": [40, 80,],
    "coherence_weight":            [1000.0, 500.0, 3000.0 ],
  
}

# ---- Fixed (non-swept) SemanticSCVI kwargs ----
N_LATENT = 10
SEMANTIC_MAX_EPOCHS = 200
SEMANTIC_BASE_KWARGS = dict(
    loss_mode="geometric",
    n_gene_sample=1024,
    n_latent=N_LATENT,
    n_layers=2,
    n_hidden=128,
    dropout_rate=0.1,
    gene_likelihood="nb",
    weights_positive=True,
    use_batch_norm=False,
)

HVG_FLAVOR = "seurat_v3"
CLUSTER_N_TOP = 500
GENE_MAPPING = ("feature_id", "feature_name")
ENABLE_LLM_GRADING = True

# Note: changing BASELINE means cache_dir `sem_baseline` refers to a different config.
# If you change BASELINE values, clear .model_cache/semantic_scvi_genept_sweep/ first.
# Same caveat for OUT_DIR/_score_cache/ when PER_PROJECTION_N_TOP changes.

In [ ]:
import sys
if str(NB_DIR) not in sys.path:
    sys.path.insert(0, str(NB_DIR))

import scanpy as sc
import torch

from benchmark_helpers import (
    _ScviAdapter,
    build_report,
    get_or_build_genept_map,
    plot_training_curves,
    train_or_load_semantic_scvi,
)
from benchmarking import SemanticBenchmark

print(f"CUDA available: {torch.cuda.is_available()}")
print(f"NB_DIR = {NB_DIR}")

_TRAIN_AXES = ("SEMANTIC_WARMUP_EPOCHS", "SEMANTIC_N_EPOCHS_KL_WARMUP", "coherence_weight")
_AXIS_SHORT = {"SEMANTIC_WARMUP_EPOCHS": "w", "SEMANTIC_N_EPOCHS_KL_WARMUP": "kl", "coherence_weight": "cw"}


def _fmt_val(v):
    if isinstance(v, float):
        return str(int(v)) if v.is_integer() else f"{v:g}".replace(".", "p")
    return str(v)


def _variant_name(variant, baseline):
    diffs = [k for k in _TRAIN_AXES if variant[k] != baseline[k]]
    if not diffs:
        return "sem_baseline"
    return "sem_" + "_".join(f"{_AXIS_SHORT[k]}{_fmt_val(variant[k])}" for k in diffs)


def _expand(baseline, sweep):
    for k in sweep:
        if k not in _TRAIN_AXES:
            raise ValueError(f"SWEEP key {k!r} not sweepable. Allowed: {_TRAIN_AXES}.")
    seen = set()
    variants = []
    def _add(v):
        t = tuple(sorted((k, v[k]) for k in _TRAIN_AXES))
        if t not in seen:
            seen.add(t)
            variants.append(dict(v))
    _add(baseline)
    for axis, values in sweep.items():
        for val in values:
            v = dict(baseline)
            v[axis] = val
            _add(v)
    return variants


variants = _expand(BASELINE, SWEEP)
HVG_TOP_N = BASELINE["HVG_TOP_N"]
PER_PROJ = BASELINE["PER_PROJECTION_N_TOP"]
print(f"HVG_TOP_N={HVG_TOP_N}  PER_PROJECTION_N_TOP={PER_PROJ}")
print(f"{len(variants)} variant(s):")
for v in variants:
    print(f"  {_variant_name(v, BASELINE):>22s}  w={v['SEMANTIC_WARMUP_EPOCHS']}  kl={v['SEMANTIC_N_EPOCHS_KL_WARMUP']}  cw={v['coherence_weight']}")

In [ ]:
adata = sc.read_h5ad(ADATA_PATH)
print("adata (raw):", adata.shape)

semantic_map = get_or_build_genept_map(adata, SEMANTIC_CACHE, GENEPT_PICKLE_PATH)
print("semantic_map (raw):", tuple(semantic_map.shape))

if HVG_TOP_N is not None:
    sc.pp.highly_variable_genes(adata, n_top_genes=HVG_TOP_N, flavor=HVG_FLAVOR, subset=False)
    keep = adata.var["highly_variable"].values
    adata = adata[:, keep].copy()
    semantic_map = semantic_map[torch.as_tensor(keep)]
    print(f"After HVG (top {HVG_TOP_N}, flavor={HVG_FLAVOR}): adata={adata.shape}, map={tuple(semantic_map.shape)}")
else:
    print("HVG filter skipped (HVG_TOP_N=None)")

In [ ]:
# Per-variant cache_dir keyed on the variant name (which encodes its diff from baseline).
# Same variant name -> cache hit; different name -> fresh train.
trained = {}  # variant_name -> (model, adata_copy)

for v in variants:
    name = _variant_name(v, BASELINE)
    cache_dir = MODEL_CACHE_DIR / SEM_CACHE_PARENT / name
    print(f"\n=== {name} === cache_dir={cache_dir}")
    adata_v = adata.copy()
    model = train_or_load_semantic_scvi(
        adata_v,
        semantic_map,
        cache_dir=cache_dir,
        force_train=False,
        max_epochs=SEMANTIC_MAX_EPOCHS,
        warmup_epochs=v["SEMANTIC_WARMUP_EPOCHS"],
        n_epochs_kl_warmup=v["SEMANTIC_N_EPOCHS_KL_WARMUP"],
        coherence_weight=v["coherence_weight"],
        **SEMANTIC_BASE_KWARGS,
    )
    trained[name] = (model, adata_v)

In [ ]:
for name, (model, _) in trained.items():
    plot_training_curves(model, name, OUT_DIR / f"training_curves_{name}.png", semantic=True)

In [ ]:
models = {name: _ScviAdapter(m, a) for name, (m, a) in trained.items()}
MODEL_NAMES = list(models.keys())
for name, m in models.items():
    print(f"{name}: latent shape = {m.get_latent_representation().shape}")

In [ ]:
bench = SemanticBenchmark(
    models, adata,
    pathway_index=str(PATHWAY_INDEX),
    gene_mapping=GENE_MAPPING,
    out_dir=str(OUT_DIR),
)
bench.run_figures(
    semantic_map=semantic_map,
    lib1_gmt=str(LIB1_GMT) if LIB1_GMT.exists() else None,
    lib2_gmt=str(LIB2_GMT) if LIB2_GMT.exists() else None,
    per_projection_n_top=PER_PROJ,
    cluster_n_top=CLUSTER_N_TOP,
)

In [ ]:
if ENABLE_LLM_GRADING:
    bench.run_grading(
        lib1_gmt=str(LIB1_GMT) if LIB1_GMT.exists() else None,
        lib2_gmt=str(LIB2_GMT) if LIB2_GMT.exists() else None,
        per_projection_n_top=PER_PROJ,
        cluster_n_top=CLUSTER_N_TOP,
    )
else:
    print("LLM grading skipped (ENABLE_LLM_GRADING=False)")

In [ ]:
from datetime import datetime
import shutil

lines = "; ".join(
    f"{_variant_name(v, BASELINE)} (w={v['SEMANTIC_WARMUP_EPOCHS']}, kl={v['SEMANTIC_N_EPOCHS_KL_WARMUP']}, cw={v['coherence_weight']})"
    for v in variants
)
notes = (
    f"SemanticSCVI sweep [GenePT-3-large]: {len(variants)} variant(s) — {lines}. "
    f"Fixed: n_latent={N_LATENT}, max_epochs={SEMANTIC_MAX_EPOCHS}, "
    f"HVG_TOP_N={HVG_TOP_N}, PER_PROJECTION_N_TOP={PER_PROJ}."
)
report_path = build_report(OUT_DIR, MODEL_NAMES, adata.shape, notes=notes)

ts = datetime.now().strftime("%Y%m%d_%H%M%S")
final_path = OUT_DIR / f"{ADATA_PATH.stem}_genept_sweep_{ts}.html"
report_path.rename(final_path)

PRESERVE_DIRS = {"_llm_cache", "_score_cache"}
for child in OUT_DIR.iterdir():
    if child == final_path:
        continue
    if child.is_dir():
        if child.name not in PRESERVE_DIRS:
            shutil.rmtree(child)
    elif child.suffix != ".html":
        child.unlink()

print(f"Report: {final_path}  ({final_path.stat().st_size / 1e6:.1f} MB)")